# Сопоставление подписантов по N-граммам

Поддерживаются форматы JSON: `Иванов А.А.`, `А.А. Иванов`, `А А Иванов`; CSV: `Иванов Андрей Александрович`.

# Сопоставление подписантов из JSON с CSV

JSON: `Иванов А.А.`; CSV: `Иванов Андрей Александрович`. Полные имя и отчество автоматически приводятся к инициалам для сравнения.

Настройте пути, два ключа JSON и столбец CSV. Используются N-граммы для фамилии (70%) и отдельное сравнение инициалов (30%).

Зависимости: `pandas`, `openpyxl`.


In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

# ===== НАСТРОЙКИ =====
JSON_DIR = Path("input_json")
CSV_FILE = "signers.csv"
OUTPUT_FILE = "result.xlsx"
JSON_KEYS = ["signer1", "signer2"]  # Укажите два ключа JSON
CSV_COLUMN = "ФИО"                 # В CSV: Иванов Иван Иванович (полностью)


def parse_fio(text):
    """Поддержка «Иванов А.А.», «А.А. Иванов» и «Иванов Андрей Александрович»."""
    if not isinstance(text, str):
        return "", ""
    parts = re.findall(r"[а-яa-z]+", text.lower().replace("ё", "е"))
    if not parts:
        return "", ""

    # Инициалы перед фамилией: А.А. Иванов / А А Иванов
    if len(parts) >= 3 and len(parts[0]) == len(parts[1]) == 1 and len(parts[2]) > 1:
        return parts[2], parts[0] + parts[1]

    # Фамилия перед инициалами или полными именем и отчеством
    return parts[0], "".join(part[0] for part in parts[1:3])


def ngrams(text, n):
    if len(text) < n:
        return {text} if text else set()
    return {text[i:i+n] for i in range(len(text)-n+1)}


def ngram_similarity(a, b):
    if not a or not b:
        return 0
    if a == b:
        return 1
    scores = []
    for n in (2, 3):
        x, y = ngrams(a, n), ngrams(b, n)
        if x and y:
            scores.append(len(x & y) / len(x | y))
    return sum(scores) / len(scores) if scores else 0


def initials_similarity(a, b):
    if not a or not b:
        return 0
    if a == b:
        return 1
    matches = sum(x == y for x, y in zip(a, b))
    return matches / max(len(a), len(b))


def similarity(fio1, fio2):
    surname1, initials1 = parse_fio(fio1)
    surname2, initials2 = parse_fio(fio2)
    if not surname1 or not surname2:
        return 0
    surname_score = ngram_similarity(surname1, surname2)
    initials_score = initials_similarity(initials1, initials2)
    if not initials1 or not initials2:
        return surname_score * 0.7
    score = surname_score * 0.7 + initials_score * 0.3
    if initials1 != initials2:
        score *= 0.7
    return score


df = pd.read_csv(CSV_FILE, sep=";", encoding="utf-8-sig")
signers = [name for name in df[CSV_COLUMN].dropna().astype(str) if name.strip()]
if not signers:
    raise ValueError("В указанном столбце CSV нет ФИО")
results = []

for file in JSON_DIR.glob("*.json"):
    with open(file, encoding="utf-8") as f:
        data = json.load(f)
    for key in JSON_KEYS:
        original = data.get(key)
        if not isinstance(original, str) or not original.strip():
            continue
        matches = [(signer, similarity(original, signer)) for signer in signers]
        best_name, best_score = max(matches, key=lambda x: x[1], default=("", 0))
        results.append({
            "Файл": file.name,
            "Ключ JSON": key,
            "Исходное ФИО": original,
            "Лучшее совпадение CSV": best_name,
            "Сходство, %": round(best_score * 100, 2),
        })

pd.DataFrame(results, columns=["Файл", "Ключ JSON", "Исходное ФИО", "Лучшее совпадение CSV", "Сходство, %"]).to_excel(OUTPUT_FILE, index=False)
print(f"Готово! Обработано записей: {len(results)}")
print(f"Результат: {OUTPUT_FILE}")
